In [7]:
import os
from IPython import get_ipython
import pandas as pd
from process_data_rf import process_twitter_data_for_rf
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_submission = pd.read_csv('data/sample_submission.csv')

In [3]:
# Data preparation - no preprocessing yet (done in CV to avoid leakage)
# Extract target from training data
y_train = train['target']

print(f"Data loaded:")
print(f"Training data shape: {train.shape}")
print(f"Test data shape: {test.shape}")
print(f"Target distribution:")
print(y_train.value_counts().sort_index())

print(f"\nTarget distribution percentages:")
target_counts = y_train.value_counts().sort_index()
for label, count in target_counts.items():
    percentage = (count / len(y_train)) * 100
    print(f"  Class {label}: {count:,} samples ({percentage:.1f}%)")

Data loaded:
Training data shape: (26206, 19)
Test data shape: (11232, 19)
Target distribution:
target
0    17423
1     8783
Name: count, dtype: int64

Target distribution percentages:
  Class 0: 17,423 samples (66.5%)
  Class 1: 8,783 samples (33.5%)


In [4]:
# XGBoost with 5-fold Cross Validation - Preprocessing at each fold to avoid data leakage
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# Initialize XGBoost classifier parameters
base_xgb_params = {
    'random_state': 42,
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'logloss'
}

print("=" * 70)
print("ROBUST 5-FOLD CROSS VALIDATION WITH FOLD-WISE PREPROCESSING")
print("=" * 70)
print("This ensures maximum robustness against data leakage by:")
print("• Processing training data separately for each fold")
print("• Using consistent feature sets across all folds")
print("• Never letting validation data influence training preprocessing")

# Prepare raw data (without any preprocessing)
raw_train_features = train.drop('target', axis=1)
y_train = train['target']

print(f"\nRaw training data shape: {raw_train_features.shape}")
print(f"Target shape: {y_train.shape}")

# 5-fold stratified cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []
fold_details = []
master_feature_list = None

print(f"\nStarting 5-fold cross validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(raw_train_features, y_train), 1):
    print(f"\n{'='*20} FOLD {fold}/5 {'='*20}")
    
    # Split raw data into train/validation for this fold
    raw_train_fold = raw_train_features.iloc[train_idx]
    raw_val_fold = raw_train_features.iloc[val_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]
    
    print(f"Raw train fold: {raw_train_fold.shape}, Raw val fold: {raw_val_fold.shape}")
    
    # Process training fold FIRST (to get feature list)
    print("Processing training fold...")
    X_train_fold, fold_feature_list = process_twitter_data_for_rf(
        raw_train_fold, verbose=False
    )
    
    # For the first fold, establish the master feature list
    if master_feature_list is None:
        master_feature_list = fold_feature_list.copy()
        print(f"Established master feature list: {len(master_feature_list)} features")
    else:
        # Verify consistency with master list
        if set(fold_feature_list) != set(master_feature_list):
            # If different, use the master list for consistency
            print(f"⚠️  Feature list differs from master - enforcing consistency")
            X_train_fold = process_twitter_data_for_rf(
                raw_train_fold, verbose=False, expected_features=master_feature_list
            )
    
    # Process validation fold with master feature list to ensure consistency
    print("Processing validation fold...")
    X_val_fold = process_twitter_data_for_rf(
        raw_val_fold, verbose=False, expected_features=master_feature_list
    )
    
    print(f"Processed train fold: {X_train_fold.shape}, Processed val fold: {X_val_fold.shape}")
    
    # Verify consistency within fold
    if X_train_fold.shape[1] != X_val_fold.shape[1]:
        print(f"❌ Feature mismatch in fold {fold}! Skipping fold.")
        continue
    
    if list(X_train_fold.columns) != list(X_val_fold.columns):
        print(f"❌ Column name mismatch in fold {fold}! Skipping fold.")
        continue
    
    print("✅ Feature consistency verified")
    
    # Train XGBoost on this fold
    print("Training XGBoost...")
    xgb_fold = xgb.XGBClassifier(**base_xgb_params)
    xgb_fold.fit(X_train_fold, y_train_fold)
    
    # Validate on this fold
    val_predictions = xgb_fold.predict(X_val_fold)
    fold_score = accuracy_score(y_val_fold, val_predictions)
    cv_scores.append(fold_score)
    
    print(f"Fold {fold} accuracy: {fold_score:.4f}")
    
    # Store fold details
    fold_details.append({
        'fold': fold,
        'train_size': len(y_train_fold),
        'val_size': len(y_val_fold),
        'accuracy': fold_score,
        'features': len(X_train_fold.columns)
    })

print(f"\n{'='*70}")
print("CROSS VALIDATION RESULTS")
print(f"{'='*70}")

cv_scores = np.array(cv_scores)
print(f"Individual fold scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"CV score range: {cv_scores.min():.4f} - {cv_scores.max():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

# Display fold details
print(f"\nFold Details:")
for detail in fold_details:
    print(f"  Fold {detail['fold']}: {detail['train_size']:,} train, {detail['val_size']:,} val, "
          f"{detail['features']} features, {detail['accuracy']:.4f} accuracy")

print(f"\nMaster feature list established: {len(master_feature_list)} features")
print(f"All folds used consistent features: ✅")

# Now train final model on full dataset using the same preprocessing approach
print(f"\n{'='*70}")
print("TRAINING FINAL MODEL ON FULL DATASET")
print(f"{'='*70}")

# Process full training data with established feature list
print("Processing full training dataset...")
X_train_final, final_feature_list = process_twitter_data_for_rf(raw_train_features, verbose=True)

# Ensure consistency with CV feature list
if set(final_feature_list) != set(master_feature_list):
    print("⚠️  Final feature list differs from CV - enforcing CV consistency")
    X_train_final = process_twitter_data_for_rf(
        raw_train_features, verbose=False, expected_features=master_feature_list
    )
    final_feature_list = master_feature_list

print("Training final XGBoost model...")
xgb_model = xgb.XGBClassifier(**base_xgb_params)
xgb_model.fit(X_train_final, y_train)

# Training accuracy
train_predictions = xgb_model.predict(X_train_final)
train_accuracy = accuracy_score(y_train, train_predictions)
print(f"Final model training accuracy: {train_accuracy:.4f}")

# Feature importance
feature_importance = xgb_model.feature_importances_
feature_names = X_train_final.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(importance_df.head(10))

# Store variables for later use
X_train = X_train_final  # For compatibility with evaluation cell
feature_list = final_feature_list  # For test data processing

ROBUST 5-FOLD CROSS VALIDATION WITH FOLD-WISE PREPROCESSING
This ensures maximum robustness against data leakage by:
• Processing training data separately for each fold
• Using consistent feature sets across all folds
• Never letting validation data influence training preprocessing

Raw training data shape: (26206, 18)
Target shape: (26206,)

Starting 5-fold cross validation...

==================== FOLD 1/5 ====================
Raw train fold: (20964, 18), Raw val fold: (5242, 18)
Processing training fold...
Established master feature list: 67 features
Processing validation fold...
Processed train fold: (20964, 67), Processed val fold: (5242, 67)
✅ Feature consistency verified
Training XGBoost...
Fold 1 accuracy: 0.8792

==================== FOLD 2/5 ====================
Raw train fold: (20965, 18), Raw val fold: (5241, 18)
Processing training fold...
⚠️  Feature list differs from master - enforcing consistency
Processing validation fold...
Processed train fold: (20965, 67), Processed

In [5]:
# Generate predictions on test set with consistent preprocessing
print("=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

print("Processing test data with consistent feature list...")
print(f"Using feature list with {len(feature_list)} features established during CV")

# Process test data using the consistent feature list from CV
X_test = process_twitter_data_for_rf(test, verbose=True, expected_features=feature_list)

print(f"\nProcessed test data shape: {X_test.shape}")
print(f"Training data shape: {X_train.shape}")

# Verify consistency
print(f"\nConsistency check:")
print(f"Same number of columns: {X_train.shape[1] == X_test.shape[1]}")
print(f"Same column names: {list(X_train.columns) == list(X_test.columns)}")

if X_train.shape[1] == X_test.shape[1] and list(X_train.columns) == list(X_test.columns):
    print("✅ Perfect feature consistency between train and test!")
else:
    print("❌ Feature inconsistency detected!")

# Generate predictions
print("\nGenerating predictions on test set...")
test_probabilities = xgb_model.predict_proba(X_test)[:, 1]

# Create submission dataframe
submission = pd.DataFrame({
    'index': test['index'],
    'target': test_probabilities
})

# Save submission file
submission.to_csv('submission_rf2.csv', index=False)

print(f"\n✅ Submission file created: submission_rf2.csv")
print(f"📊 Prediction statistics:")
print(f"  Range: {test_probabilities.min():.4f} to {test_probabilities.max():.4f}")
print(f"  Mean:  {test_probabilities.mean():.4f}")
print(f"  Std:   {test_probabilities.std():.4f}")
print(f"  Shape: {submission.shape}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(submission.head(10))

print(f"\n🚀 Ready for submission with robust preprocessing!")
print("   • No data leakage: ✅")
print("   • Consistent features: ✅") 
print("   • Proper probabilities: ✅")

GENERATING TEST PREDICTIONS
Processing test data with consistent feature list...
Using feature list with 67 features established during CV
=== Random Forest Data Preprocessing ===
Original dataframe shape: (11232, 19)
Numeric features (6): ['favourites_count', 'followers_count', 'friends_count', 'average_tweets_per_day', 'account_age_days', 'statuses_count']
Binary features (4): ['default_profile', 'default_profile_image', 'geo_enabled', 'verified']
Categorical features (1): ['lang']
Dropping 7 features: ['id', 'created_at', 'location', 'description', 'screen_name', 'profile_background_image_url', 'profile_image_url']

Processing numerical features...

Processing binary features

Processing categorical features...
  - lang: filled 2369 missing values with 'unknown'
  - lang: 47 -> 47 categories (grouped 0 with freq < 0)
  - lang: created 47 one-hot encoded columns

Processing textual features...
  - location_len: mean=11.20, max=110
  - description_len: mean=67.09, max=190
  - screen_n

In [9]:
# Comprehensive Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

print("=" * 60)
print("COMPREHENSIVE MODEL EVALUATION")
print("=" * 60)

# Generate predictions and probabilities
train_pred_proba = xgb_model.predict_proba(X_train)[:, 1]
train_predictions = xgb_model.predict(X_train)

# 1. Classification Report
print("\n1. CLASSIFICATION REPORT (Training Data):")
print("-" * 40)
print(classification_report(y_train, train_predictions))

# 2. Confusion Matrix
print("2. CONFUSION MATRIX (Training Data):")
print("-" * 40)
cm = confusion_matrix(y_train, train_predictions)
print(f"Confusion Matrix:\n{cm}")

# Extract values for better interpretation
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"True Negatives (TN):  {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives (TP):  {tp:,}")

# 3. Additional Metrics
print("\n3. DETAILED METRICS:")
print("-" * 40)
from sklearn.metrics import precision_score, recall_score, f1_score

accuracy = accuracy_score(y_train, train_predictions)
precision = precision_score(y_train, train_predictions)
recall = recall_score(y_train, train_predictions)
f1 = f1_score(y_train, train_predictions)
roc_auc = roc_auc_score(y_train, train_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

# 4. Calculate Average Precision (PR AUC)
avg_precision = average_precision_score(y_train, train_pred_proba)
print(f"PR AUC:    {avg_precision:.4f}")

# 5. Class Distribution Analysis
print("\n4. CLASS DISTRIBUTION ANALYSIS:")
print("-" * 40)
class_counts = y_train.value_counts().sort_index()
print("True class distribution:")
for class_label, count in class_counts.items():
    percentage = (count / len(y_train)) * 100
    print(f"  Class {class_label}: {count:,} samples ({percentage:.1f}%)")

print("\nPredicted class distribution:")
pred_counts = pd.Series(train_predictions).value_counts().sort_index()
for class_label, count in pred_counts.items():
    percentage = (count / len(train_predictions)) * 100
    print(f"  Class {class_label}: {count:,} samples ({percentage:.1f}%)")

# 6. Probability Distribution Analysis
print("\n5. PROBABILITY DISTRIBUTION ANALYSIS:")
print("-" * 40)
print(f"Probability statistics:")
print(f"  Min probability:  {train_pred_proba.min():.4f}")
print(f"  Max probability:  {train_pred_proba.max():.4f}")
print(f"  Mean probability: {train_pred_proba.mean():.4f}")
print(f"  Std probability:  {train_pred_proba.std():.4f}")

# Analyze predictions by class
bot_probs = train_pred_proba[y_train == 1]
human_probs = train_pred_proba[y_train == 0]

print(f"\nProbability by true class:")
print(f"  Bots (Class 1) - Mean: {bot_probs.mean():.4f}, Std: {bot_probs.std():.4f}")
print(f"  Humans (Class 0) - Mean: {human_probs.mean():.4f}, Std: {human_probs.std():.4f}")

# 7. Model Performance Summary
print("\n6. MODEL PERFORMANCE SUMMARY:")
print("-" * 40)
print("Cross-Validation Performance:")
print(f"  Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"  CV Score Range: {cv_scores.min():.4f} - {cv_scores.max():.4f}")

print(f"\nTraining Performance:")
print(f"  Training Accuracy: {accuracy:.4f}")
print(f"  Training ROC AUC:  {roc_auc:.4f}")

# Check for potential overfitting
cv_mean = cv_scores.mean()
overfitting_diff = accuracy - cv_mean
print(f"\nOverfitting Check:")
print(f"  CV vs Train Accuracy Difference: {overfitting_diff:.4f}")
if overfitting_diff > 0.05:
    print("  ⚠️  Potential overfitting detected (difference > 0.05)")
elif overfitting_diff > 0.02:
    print("  ⚠️  Mild overfitting detected (difference > 0.02)")
else:
    print("  ✅ No significant overfitting detected")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)

COMPREHENSIVE MODEL EVALUATION

1. CLASSIFICATION REPORT (Training Data):
----------------------------------------
              precision    recall  f1-score   support

           0       0.91      0.96      0.93     17423
           1       0.90      0.82      0.86      8783

    accuracy                           0.91     26206
   macro avg       0.91      0.89      0.90     26206
weighted avg       0.91      0.91      0.91     26206

2. CONFUSION MATRIX (Training Data):
----------------------------------------
Confusion Matrix:
[[16649   774]
 [ 1581  7202]]

Confusion Matrix Breakdown:
True Negatives (TN):  16,649
False Positives (FP): 774
False Negatives (FN): 1,581
True Positives (TP):  7,202

3. DETAILED METRICS:
----------------------------------------
Accuracy:  0.9101
Precision: 0.9030
Recall:    0.8200
F1-Score:  0.8595
ROC AUC:   0.9648
PR AUC:    0.9429

4. CLASS DISTRIBUTION ANALYSIS:
----------------------------------------
True class distribution:
  Class 0: 17,423 sam

In [ ]:
# Plot feature importances (uses importance_df created earlier)
# Show top N features as a horizontal bar chart
top_n = 68
top_df = importance_df.head(top_n).iloc[::-1]  # reverse for plotting

plt.figure(figsize=(8, 0.4 * len(top_df) + 2))
plt.barh(top_df['feature'], top_df['importance'], color='C0')
plt.xlabel('Importance')
plt.title(f'Top {top_n} Feature Importances')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [10]:
# Grid Search function with fold-wise preprocessing for better data leakage prevention
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin

class XGBoostWithPreprocessing(BaseEstimator):
    """
    Custom estimator that combines preprocessing and XGBoost training.
    This ensures preprocessing happens within each CV fold to prevent data leakage.
    """
    
    def __init__(self, n_estimators=100, max_depth=6, learning_rate=0.1, 
                 subsample=0.8, colsample_bytree=0.8, min_child_weight=1, 
                 random_state=42, eval_metric='logloss'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.min_child_weight = min_child_weight
        self.random_state = random_state
        self.eval_metric = eval_metric
        self.feature_list_ = None
        self.model_ = None
    
    def fit(self, X_raw, y):
        """
        Fit the model with preprocessing applied to raw data.
        """
        # Process raw training data to get features and feature list
        X_processed, self.feature_list_ = process_twitter_data_for_rf(X_raw, verbose=False)
        
        # Initialize and train XGBoost model
        self.model_ = xgb.XGBClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            min_child_weight=self.min_child_weight,
            random_state=self.random_state,
            eval_metric=self.eval_metric
        )
        
        self.model_.fit(X_processed, y)
        return self
    
    def predict(self, X_raw):
        """
        Make predictions on raw data (applies preprocessing first).
        """
        X_processed = process_twitter_data_for_rf(
            X_raw, verbose=False, expected_features=self.feature_list_
        )
        return self.model_.predict(X_processed)
    
    def predict_proba(self, X_raw):
        """
        Make probability predictions on raw data (applies preprocessing first).
        """
        X_processed = process_twitter_data_for_rf(
            X_raw, verbose=False, expected_features=self.feature_list_
        )
        return self.model_.predict_proba(X_processed)
    
    def get_params(self, deep=True):
        """Get parameters for GridSearchCV."""
        return {
            'n_estimators': self.n_estimators,
            'max_depth': self.max_depth,
            'learning_rate': self.learning_rate,
            'subsample': self.subsample,
            'colsample_bytree': self.colsample_bytree,
            'min_child_weight': self.min_child_weight,
            'random_state': self.random_state,
            'eval_metric': self.eval_metric
        }
    
    def set_params(self, **params):
        """Set parameters for GridSearchCV."""
        for key, value in params.items():
            setattr(self, key, value)
        return self

def perform_robust_xgboost_grid_search(raw_train_features, y_train, cv_folds=5):
    """
    Perform grid search for XGBoost with fold-wise preprocessing to prevent data leakage.
    
    Parameters:
    raw_train_features: Raw training features (before preprocessing)
    y_train: Training target
    cv_folds: Number of cross-validation folds
    
    Returns:
    best_model: Best XGBoost model from grid search
    best_params: Best parameters found
    best_score: Best cross-validation score
    """
    
    print("Starting ROBUST XGBoost Grid Search with fold-wise preprocessing...")
    print("This prevents data leakage by preprocessing within each CV fold.")
    print("This may take longer but provides more reliable results.")
    
    # Define parameter grid - you can modify these based on your needs
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5]
    }
    
    # For faster testing, you can use a smaller grid:
    # param_grid = {
    #     'n_estimators': [100, 200],
    #     'max_depth': [4, 6],
    #     'learning_rate': [0.1, 0.2]
    # }
    
    # Initialize custom estimator with preprocessing
    estimator = XGBoostWithPreprocessing(random_state=42, eval_metric='logloss')
    
    # Setup cross-validation
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    # Initialize GridSearchCV
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring='accuracy',
        cv=skf,
        n_jobs=-1,  # Use all available cores
        verbose=1   # Show progress
    )
    
    # Perform grid search
    total_combinations = len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['learning_rate']) * len(param_grid['subsample']) * len(param_grid['colsample_bytree']) * len(param_grid['min_child_weight'])
    print(f"Testing {total_combinations} parameter combinations with {cv_folds}-fold CV...")
    print(f"Total model training runs: {total_combinations * cv_folds}")
    
    grid_search.fit(raw_train_features, y_train)
    
    # Get results
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    
    print(f"\nRobust Grid Search Complete!")
    print(f"Best Cross-Validation Score: {best_score:.4f}")
    print(f"Best Parameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    
    # Get feature importance from the best model (need to access the internal XGBoost model)
    if hasattr(best_model, 'model_') and best_model.model_ is not None:
        feature_importance = best_model.model_.feature_importances_
        feature_names = best_model.feature_list_
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 10 Most Important Features (Best Model):")
        print(importance_df.head(10))
    
    return best_model, best_params, best_score

# Usage example:
print("="*70)
print("ROBUST GRID SEARCH - Uses fold-wise preprocessing like Cell 4")
print("="*70)
print("This version processes data within each CV fold to prevent data leakage")
print("Use this for maximum robustness (same approach as your 5-fold CV)")
print()

# Prepare raw data for grid search (same as Cell 4)
raw_train_features = train.drop('target', axis=1)
y_train = train['target']

print("Starting robust grid search...")
best_model_robust, best_params_robust, best_score_robust = perform_robust_xgboost_grid_search(
    raw_train_features, y_train, cv_folds=5
)

# Generate improved predictions with best model
print("\nGenerating predictions with robust best model...")
test_probabilities_robust = best_model_robust.predict_proba(test)[:, 1]

# Create improved submission
submission_robust = pd.DataFrame({
    'index': test['index'],
    'target': test_probabilities_robust
})

submission_robust.to_csv('submission_rf2_gs.csv', index=False)
print("Robust submission saved as: submission_rf2_gs.csv")

ROBUST GRID SEARCH - Uses fold-wise preprocessing like Cell 4
This version processes data within each CV fold to prevent data leakage
Use this for maximum robustness (same approach as your 5-fold CV)

Starting robust grid search...
Starting ROBUST XGBoost Grid Search with fold-wise preprocessing...
This prevents data leakage by preprocessing within each CV fold.
This may take longer but provides more reliable results.
Testing 729 parameter combinations with 5-fold CV...
Total model training runs: 3645
Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Exception ignored on calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x10d8ebcd0>>:
Traceback (most recent call last):
  File "/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/.venv/lib/python3.13/site-packages/xgboost/core.py", line 585, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


KeyboardInterrupt: 